In [1]:
from sentence_transformers import CrossEncoder,SentenceTransformer,util

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(["How do I reset my password?", "Steps to change my login credentials"])
# compare with cosine similarity

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
len(embeddings[0])

384

In [10]:
## score using Bi-Endoder
emb_a = model.encode("How do I reset my password?")        # Step 1: text → embedding
emb_b = model.encode("Steps to change my login credentials")  # Step 1: text → embedding

score_biencoder = util.cos_sim(emb_a, emb_b)                          # Step 2: math → score

In [11]:
score_biencoder

tensor([[0.5837]])

In [3]:
## score using CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict([("How do I reset my password?", "Steps to change your login credentials")])

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

C:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prati\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\hugg

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [7]:
scores

array([-6.553713], dtype=float32)

## using quadrant inbuild funtionality

In [3]:
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding
from qdrant_client.models import Distance, VectorParams, PointStruct, models
from qdrant_client import QdrantClient


In [3]:
dense_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
print("✓ Dense embedding model loaded (all-MiniLM-L6-v2)")

sparse_model = SparseTextEmbedding("Qdrant/bm25")
print("✓ Sparse embedding model loaded (BM25)")

late_interaction_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")
print("✓ Late interaction model loaded (ColBERTv2.0)")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

C:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prati\AppData\Local\Temp\fastembed_cache. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: 

✓ Dense embedding model loaded (all-MiniLM-L6-v2)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

... (2 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

C:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prati\AppData\Local\Temp\fastembed_cache\models--colbert-ir--colbertv2.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


✓ Late interaction model loaded (ColBERTv2.0)


In [5]:
query = "What connected load or contract demand makes a commercial building fall within the scope of ECBC 2017?"


In [7]:
client = QdrantClient(path='D:\\M.tech\\Research Area\\RAG pipeline\\DB\\bge-small-en-v1.5_Embedding\\StructureAwareChunking')

In [10]:
"""
Execute hybrid search with reranking.
1. Prefetch results using dense (semantic) and sparse (keyword) search
2. Rerank combined results using ColBERT late interaction
"""
# Generate query embeddings
dense_query_vector = list(dense_model.query_embed(query))[0]
sparse_query_vector = list(sparse_model.query_embed(query))[0]
late_interaction_query_vector = list(late_interaction_model.query_embed(query))[0]

# Set up prefetch for hybrid search
prefetch = [
    models.Prefetch(
        query=dense_query_vector,
        using=None,
        limit=20,
    ),
    models.Prefetch(
        query=models.SparseVector(**sparse_query_vector.as_object()),
        using="bm25",
        limit=20,
    ),
]

# Execute search with ColBERT reranking
results = client.query_points(
    collection_name="bge_small_en_v1.5_StructureAwareChunking",
    prefetch=prefetch,
    query=late_interaction_query_vector,
    using="colbertv2.0",
    with_payload=True,
    limit=10,
)



ValueError: Sparse vector bm25 is not found in the collection